# 第2回　確率論の基礎(1)：条件付き確率
## ―― 「情報を得る」とは「確率を更新する」こと。そして今期の主役「独立性」を定義する

統計学Ⅱ　2026後期　／　北星学園大学　／　小野原 彩香

---

### このノートの使い方

今日も ▶ を上から押して、**自分の予想とデータの答えを見比べる**。

今日のキーワードは2つ ―― **条件付き確率 $P(A\mid B)$** と **独立**。
とくに「独立」は、第13回「コンドルセの陪審定理」まで今期をずっと貫く最重要概念だ。ここでしっかり数値で掴んでおこう。

In [ ]:
# 準備。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
print("準備OK。次のセルへ。")

---
## 1. 直感クイズ ―― 2人の子ども問題

ある家庭に子どもが **2人** いる。性別は男・女が半々で、それぞれ独立に決まったとする。

いま、**「少なくとも1人は女の子」** だと分かっている。

**問い：このとき、2人とも女の子である確率は？**

多くの人は「もう1人が女の子か男の子か、だから 1/2」と感じる。

予想を決めてから ▶。

In [ ]:
# 2人きょうだいの家庭を10万世帯つくってシミュレーション
rng = np.random.default_rng(0)
N = 100_000
上の子 = rng.integers(0, 2, N)   # 0=男, 1=女
下の子 = rng.integers(0, 2, N)

# 条件：少なくとも1人は女の子
少なくとも1人女 = (上の子 == 1) | (下の子 == 1)
# 注目：2人とも女の子
両方女 = (上の子 == 1) & (下の子 == 1)

# P(両方女 | 少なくとも1人女) = 両方成り立つ世帯 ÷ 条件を満たす世帯
条件付き確率 = 両方女[少なくとも1人女].mean()
print(f"『少なくとも1人は女』の世帯数： {少なくとも1人女.sum():,}")
print(f"そのうち『2人とも女』の割合 ＝ P(2人とも女 | 少なくとも1人女) ＝ {条件付き確率:.3f}")

**答えは 1/2 ではなく、約 1/3。**

理由：2人の組み合わせは（上,下）＝（男,男）（男,女）（女,男）（女,女）の4通りで、最初はすべて等確率。
「少なくとも1人は女」という情報で **（男,男）が消える** ので、残りは3通り。そのうち「2人とも女」は1通り。だから **1/3**。

> 「少なくとも1人は女」と聞いた瞬間、確率が $1/4 \to 1/3$ に **更新** された。これが条件付き確率だ。

---
## 2. 条件付き確率とは

事象 $B$ が起きたと分かったうえでの $A$ の確率を、**条件付き確率** $P(A\mid B)$ と書く。

$$P(A\mid B) = \frac{P(A \cap B)}{P(B)}$$

「$B$ という条件の世界に絞り込み、その中で $A$ が占める割合」を測っている。さっきのコードの
`2人とも女[少なくとも1人女].mean()` は、まさに **「条件を満たす世帯だけに絞って、その中の割合」** を計算していた。

**情報を得る ＝ 標本空間を絞り込む ＝ 確率を更新する。** これが今期ずっと使う考え方だ（第3回ベイズへ続く）。

---
## 3. 今期の主役 ―― 「独立」

ここで、今期もっとも大事な概念を定義する。

事象 $A$ と $B$ が **独立** であるとは ―― **$B$ を知っても $A$ の確率が変わらない** こと。

$$P(A\mid B) = P(A) \qquad(\text{同じことだが } P(A\cap B)=P(A)\,P(B))$$

逆に、$B$ を知ると $A$ の確率が変わるなら、2つは **従属（独立でない）**。

次のセルで、**独立な例** と **従属な例** を実際に作って、$P(A\mid B)$ と $P(A)$ を見比べよう。

In [ ]:
# 例A：2回のコイン投げ（独立）。 A=2投目が表, B=1投目が表
rng = np.random.default_rng(1)
M = 200_000
一投目 = rng.integers(0, 2, M)
二投目 = rng.integers(0, 2, M)
PA_indep         = (二投目 == 1).mean()
PA_given_B_indep = (二投目[一投目 == 1] == 1).mean()
print("【独立な例：2回のコイン投げ】")
print(f"  P(2投目=表)              = {PA_indep:.3f}")
print(f"  P(2投目=表 | 1投目=表)   = {PA_given_B_indep:.3f}")
print("  → ほぼ同じ。1投目を知っても2投目の確率は変わらない＝独立")

In [ ]:
# 例B：トランプ52枚から続けて2枚引く・戻さない（従属）。 A=2枚目がハート, B=1枚目がハート
# ハート13枚/52枚。1枚目を引いた後、残り51枚で2枚目を引く。
rng = np.random.default_rng(2)
K = 200_000
# 山札：先頭13枚をハート(True)とし、各試行ごとにシャッフルして上2枚を見る
base = np.zeros(52, dtype=bool); base[:13] = True
deck = np.tile(base, (K, 1))
idx = np.argsort(rng.random((K, 52)), axis=1)      # 各行をばらばらに並べ替え＝シャッフル
deck = np.take_along_axis(deck, idx, axis=1)
一枚目H = deck[:, 0]
二枚目H = deck[:, 1]
PA_dep         = 二枚目H.mean()
PA_given_B_dep = 二枚目H[一枚目H].mean()
print("【従属な例：トランプを戻さず2枚引く】")
print(f"  P(2枚目=ハート)               = {PA_dep:.3f}   （理論 13/52 = 0.250）")
print(f"  P(2枚目=ハート | 1枚目=ハート) = {PA_given_B_dep:.3f}   （理論 12/51 ≈ 0.235）")
print("  → 下がった。1枚目を知ると2枚目の確率が変わる＝従属（独立でない）")

**独立の例ではグラフが動かず、従属の例では動く。** この「条件を知ると確率が変わるか？」が独立かどうかの分かれ目だ。

In [ ]:
# 2つの例を並べて棒グラフで比較
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

独立値 = [PA_indep, PA_given_B_indep]
axes[0].bar(["P(A)", "P(A|B)"], 独立値, color=["#888", "#3949ab"])
axes[0].set_title("独立：コイン2回\n条件を知っても変わらない")
axes[0].set_ylim(0, 0.6)
for i, v in enumerate(独立値):
    axes[0].text(i, v + 0.01, f"{v:.3f}", ha="center")

従属値 = [PA_dep, PA_given_B_dep]
axes[1].bar(["P(A)", "P(A|B)"], 従属値, color=["#888", "#e8503a"])
axes[1].set_title("従属：トランプ2枚\n条件を知ると下がる")
axes[1].set_ylim(0, 0.30)
for i, v in enumerate(従属値):
    axes[1].text(i, v + 0.005, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.show()

---
## 4. なぜ「独立」が今期の主役なのか

統計学Ⅱでこれから学ぶ道具は、ほとんどが **「独立」を前提** にしている。

- 第4回 **二項分布**：独立なコイン投げを n 回くりかえす
- 第5回 **正規分布**：独立な多数の要因の和
- 第6回 **中心極限定理**：独立な標本の平均
- 第7回 **区間推定**：独立な標本だから誤差が計算できる

そして第12〜13回で、人々が「**空気を読んで**」互いの判断を真似し始めると、この **独立が壊れ**、集団の判断が劣化することを見る。

> 独立とは「他に影響されず、自分の情報だけで決まっている」状態。これが壊れると、統計の土台そのものが揺らぐ。今日その定義を手に入れた。

---
## 5. 注意 ―― 「独立」と「排反（同時に起きない）」は別物

よくある混同。サイコロで $A$=「偶数の目」、$B$=「3の目」。

- $A$ と $B$ は **排反**（同時には起きない。3は偶数でない）
- だが **独立ではない**：$B$（3が出た）と分かれば $A$（偶数）の確率は $1/2 \to 0$ に変わる

**排反は「重ならない」、独立は「影響しない」。** まったく違う概念なので注意。次のセルで確かめる。

In [ ]:
rng = np.random.default_rng(3)
さいころ = rng.integers(1, 7, 300_000)
A_偶数 = np.isin(さいころ, [2, 4, 6])
B_3   = (さいころ == 3)
print(f"P(A=偶数)            = {A_偶数.mean():.3f}")
print(f"P(A=偶数 | B=3が出た) = {A_偶数[B_3].mean():.3f}   ← 3は偶数でないので 0 に変わる")
print("→ 排反（重ならない）だが、条件で確率が変わるので『独立ではない』")

---
## 今日のまとめ

| 概念 | 意味 | 式 |
|---|---|---|
| 条件付き確率 | $B$ を知った後の $A$ の確率。情報で確率を更新 | $P(A\mid B)=\dfrac{P(A\cap B)}{P(B)}$ |
| 独立 | $B$ を知っても $A$ の確率が変わらない | $P(A\mid B)=P(A)$ |
| 排反 | $A$ と $B$ は同時に起きない（≠独立） | $P(A\cap B)=0$ |

- 「少なくとも1人は女」で確率が $1/2 \to 1/3$ に **更新** された＝条件付き確率。
- 独立とは「影響されない」こと。今期の道具（二項・正規・CLT・区間推定）はすべて独立が前提。
- 「空気を読む」＝互いに影響し合う＝独立を壊す、という今期の核心への入口に立った。

> **課題（Moodle）**：条件付き確率の計算（自動採点）＋「独立とは何か。独立でない身近な例を1つ挙げ、なぜ独立でないか」の記述。詳しくはMoodleの第2回課題を見ること。

> **次回予告**：第3回「ベイズの定理の深掘り」。「精度99%の検査で陽性。本当に病気の確率は？」直感は99%、正解は数%。条件付き確率を逆向き $P(\text{病気}\mid\text{陽性})$ に使う技術を学ぶ。